- **Name:** 07_dataframe_dropDuplicates
- **Author:** Shamas Imran
- **Desciption:** Removing duplicate rows from DataFrames
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Applied dropDuplicates on single column  
                                              Used subset for multi-column deduplication  
                                              Compared distinct vs dropDuplicates  
-->

In [1]:

data = [
    (1, "Alice", 20, "2024-01-01"),
    (1, "Alice", 20, "2024-01-01"),  # duplicate of row 1
    (3, "Bob",   25, "2024-01-02"),
    (4, "Bob",   25, "2024-01-03"),  # duplicate on name+age but diff date
    (5, "Eve",   30, "2024-01-04"),
]
columns = ["ID", "Name", "Age", "JoinDate"]
df = spark.createDataFrame(data, columns)
df.show()

StatementMeta(, 25530b61-5338-430a-9680-618c312c21ef, 3, Finished, Available, Finished)

+---+-----+---+----------+
| ID| Name|Age|  JoinDate|
+---+-----+---+----------+
|  1|Alice| 20|2024-01-01|
|  1|Alice| 20|2024-01-01|
|  3|  Bob| 25|2024-01-02|
|  4|  Bob| 25|2024-01-03|
|  5|  Eve| 30|2024-01-04|
+---+-----+---+----------+



In [2]:
df_no_dupes = df.dropDuplicates()
# Removes row 2 since it’s identical to row 1.
display(df_no_dupes)

StatementMeta(, 25530b61-5338-430a-9680-618c312c21ef, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 163ea91e-2e72-4a2b-b592-209f63c47491)

In [3]:
df_unique_name_age = df.dropDuplicates(["Name", "Age"])
df_unique_name_age.show()

StatementMeta(, 25530b61-5338-430a-9680-618c312c21ef, 5, Finished, Available, Finished)

+---+-----+---+----------+
| ID| Name|Age|  JoinDate|
+---+-----+---+----------+
|  1|Alice| 20|2024-01-01|
|  3|  Bob| 25|2024-01-02|
|  5|  Eve| 30|2024-01-04|
+---+-----+---+----------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

windowSpec = Window.partitionBy("Name").orderBy(df["JoinDate"].desc())

df_latest = (
    df.withColumn("row_num", row_number().over(windowSpec))
      .filter("row_num = 1")
      .drop("row_num")
)

df_latest.show()